<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/02_ML_Engineer/intermediate/05_model_monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Monitoring & Drift Detection

Production models degrade silently. This notebook builds a complete monitoring stack — detect data drift, measure model quality decay, and auto-trigger retraining before users notice.

**Topics:** Data drift (KS, PSI, Chi²), concept drift, prediction drift, Evidently AI reports, statistical process control, alerting, automated retraining triggers

## 1. Why Models Degrade — The Three Types of Drift

In [ ]:
import numpy as np
from scipy import stats
from dataclasses import dataclass

@dataclass
class DriftReport:
    feature: str
    method: str
    statistic: float
    p_value: float
    drift_detected: bool
    severity: str  # low / medium / high

def ks_drift(ref: np.ndarray, cur: np.ndarray, alpha: float = 0.05, feature: str = '') -> DriftReport:
    """Kolmogorov-Smirnov test for continuous feature drift."""
    stat, p = stats.ks_2samp(ref, cur)
    severity = 'high' if p < 0.001 else 'medium' if p < alpha else 'low'
    return DriftReport(feature, 'KS', stat, p, p < alpha, severity)

def psi_drift(ref: np.ndarray, cur: np.ndarray, n_bins: int = 10, feature: str = '') -> DriftReport:
    """Population Stability Index. PSI > 0.25 = significant drift."""
    bins = np.percentile(ref, np.linspace(0, 100, n_bins + 1))
    bins[0] -= 1e-10; bins[-1] += 1e-10
    ref_pct = np.histogram(ref, bins=bins)[0] / len(ref)
    cur_pct = np.histogram(cur, bins=bins)[0] / len(cur)
    ref_pct = np.where(ref_pct == 0, 1e-6, ref_pct)
    cur_pct = np.where(cur_pct == 0, 1e-6, cur_pct)
    psi = float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))
    drift = psi > 0.25
    severity = 'high' if psi > 0.25 else 'medium' if psi > 0.1 else 'low'
    return DriftReport(feature, 'PSI', psi, 0.0, drift, severity)

def chi2_drift(ref: np.ndarray, cur: np.ndarray, feature: str = '') -> DriftReport:
    """Chi-square test for categorical feature drift."""
    all_cats = np.union1d(ref, cur)
    ref_counts = np.array([np.sum(ref == c) for c in all_cats]) + 1  # Laplace smoothing
    cur_counts = np.array([np.sum(cur == c) for c in all_cats]) + 1
    # Chi2 test: are proportions different?
    ref_norm = ref_counts / ref_counts.sum() * cur_counts.sum()
    stat, p = stats.chisquare(cur_counts, f_exp=ref_norm)
    severity = 'high' if p < 0.001 else 'medium' if p < 0.05 else 'low'
    return DriftReport(feature, 'Chi2', stat, p, p < 0.05, severity)

# Simulate reference (training) and production data
np.random.seed(42)
n_ref, n_cur = 5000, 1000

reference = {
    'age':        np.random.normal(35, 10, n_ref),
    'income':     np.random.lognormal(10.5, 0.6, n_ref),
    'credit_score': np.random.normal(680, 80, n_ref),
    'job_type':   np.random.choice(['salaried', 'self_employed', 'unemployed'], n_ref, p=[0.6, 0.3, 0.1]),
}
# Simulate 6 months later: age shifted up, income stable, job mix changed
production = {
    'age':        np.random.normal(38, 11, n_cur),      # ← slight drift
    'income':     np.random.lognormal(10.5, 0.6, n_cur), # stable
    'credit_score': np.random.normal(640, 100, n_cur),   # ← significant drift!
    'job_type':   np.random.choice(['salaried', 'self_employed', 'unemployed'], n_cur, p=[0.4, 0.4, 0.2]),  # shifted
}

print('Feature Drift Analysis:')
print(f'{"Feature":<15} {"Method":<8} {"Stat":>10} {"p-value":>10} {"Drift":>7} {"Severity"}')
print('-' * 65)

for feat in ['age', 'income', 'credit_score']:
    r = ks_drift(reference[feat], production[feat], feature=feat)
    print(f'{r.feature:<15} {r.method:<8} {r.statistic:>10.4f} {r.p_value:>10.4f} {str(r.drift_detected):>7} {r.severity}')
    r2 = psi_drift(reference[feat], production[feat], feature=feat)
    print(f'{"":<15} {r2.method:<8} {r2.statistic:>10.4f} {"N/A":>10} {str(r2.drift_detected):>7} {r2.severity}')

r3 = chi2_drift(reference['job_type'], production['job_type'], 'job_type')
print(f'{r3.feature:<15} {r3.method:<8} {r3.statistic:>10.4f} {r3.p_value:>10.4f} {str(r3.drift_detected):>7} {r3.severity}')

## 2. Prediction Drift and Model Performance Decay

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from scipy import stats

np.random.seed(42)

# Train a model on historical data
def make_dataset(n: int, concept_shift: float = 0.0) -> tuple[np.ndarray, np.ndarray]:
    """Generate loan default dataset. concept_shift simulates relationship change."""
    X = np.column_stack([
        np.random.normal(35, 10, n),         # age
        np.random.normal(50_000, 20_000, n), # income
        np.random.normal(680, 80, n),        # credit score
    ])
    # True relationship changes over time (concept drift)
    log_odds = (
        -0.02 * X[:, 0] +
        -0.00002 * X[:, 1] +
        -0.01 * X[:, 2] +
        concept_shift * X[:, 0] +  # age becomes less predictive
        np.random.normal(0, 0.5, n)
    )
    y = (log_odds > np.median(log_odds)).astype(int)
    return X, y

# Train on historical data
X_train, y_train = make_dataset(5000, concept_shift=0.0)
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Simulate monthly performance monitoring
print('Model Performance Over Time (simulating concept drift):')
print(f'{"Month":<8} {"AUC":>8} {"Pred Drift (PSI)":>18} {"Status"}')
print('-' * 50)

for month in range(1, 9):
    # Concept drifts gradually starting month 3
    shift = max(0, (month - 2) * 0.003)
    X_prod, y_prod = make_dataset(500, concept_shift=shift)

    # AUC (requires labels — delayed by ~30 days in practice)
    y_pred = model.predict_proba(X_prod)[:, 1]
    auc = roc_auc_score(y_prod, y_pred)

    # Prediction drift (no labels needed — monitor in real time)
    ref_preds = model.predict_proba(X_train[:500])[:, 1]
    psi_result = psi_drift(ref_preds, y_pred, feature='prediction_score')

    status = 'ALERT' if auc < 0.70 or psi_result.drift_detected else ('WATCH' if psi_result.statistic > 0.1 else 'OK')
    print(f'M{month:<7} {auc:>8.4f} {psi_result.statistic:>18.4f} {status}')

print()
print('Key insight: Prediction drift (PSI) detects problems BEFORE labels arrive.')
print('In production: monitor predictions daily, evaluate AUC weekly when labels arrive.')

## 3. Statistical Process Control — CUSUM and EWMA Alerts

In [ ]:
import numpy as np

class CUSUMDetector:
    """Cumulative Sum (CUSUM) change point detection.
    Detects sustained shifts in a metric stream."""

    def __init__(self, k: float = 0.5, h: float = 4.0, target: float = None):
        """
        k: allowance (detect shift larger than k std devs)
        h: decision threshold (alert when CUSUM > h)
        target: expected value under control (estimated from warmup if None)
        """
        self.k = k
        self.h = h
        self.target = target
        self.S_high = 0.0   # Upward CUSUM
        self.S_low = 0.0    # Downward CUSUM
        self._warmup: list[float] = []
        self._std: float = 1.0

    def update(self, value: float) -> bool:
        """Update CUSUM. Returns True if change point detected."""
        if self.target is None:
            self._warmup.append(value)
            if len(self._warmup) >= 30:
                self.target = np.mean(self._warmup)
                self._std = max(np.std(self._warmup), 1e-6)
            return False

        z = (value - self.target) / self._std
        self.S_high = max(0, self.S_high + z - self.k)
        self.S_low  = max(0, self.S_low  - z - self.k)
        return self.S_high > self.h or self.S_low > self.h

    def reset(self):
        self.S_high = 0.0
        self.S_low = 0.0

class EWMADetector:
    """Exponentially Weighted Moving Average for metric monitoring."""

    def __init__(self, alpha: float = 0.2, n_sigma: float = 3.0):
        self.alpha = alpha
        self.n_sigma = n_sigma
        self._ewma: float = None
        self._ewma_var: float = None

    def update(self, value: float) -> tuple[float, bool]:
        if self._ewma is None:
            self._ewma = value
            self._ewma_var = 0.0
            return value, False
        self._ewma = self.alpha * value + (1 - self.alpha) * self._ewma
        self._ewma_var = self.alpha * (value - self._ewma)**2 + (1 - self.alpha) * self._ewma_var
        std = max(np.sqrt(self._ewma_var), 1e-6)
        alert = abs(value - self._ewma) > self.n_sigma * std
        return self._ewma, alert

# Simulate AUC degradation stream
np.random.seed(42)
normal_aucs = np.random.normal(0.85, 0.01, 50)  # Stable period
degraded_aucs = np.random.normal(0.78, 0.01, 30)  # Gradual drop
auc_stream = np.concatenate([normal_aucs, degraded_aucs])

cusum = CUSUMDetector(k=0.5, h=4.0)
ewma = EWMADetector(alpha=0.3, n_sigma=3.0)

print('CUSUM + EWMA Monitoring on AUC stream (degradation starts at step 50):')
print(f'{"Step":<6} {"AUC":>8} {"CUSUM_hi":>10} {"CUSUM_lo":>10} {"EWMA":>8} {"Alert"}')
print('-' * 60)

for i, auc in enumerate(auc_stream):
    cusum_alert = cusum.update(auc)
    ewma_val, ewma_alert = ewma.update(auc)
    alert = cusum_alert or ewma_alert
    if i % 5 == 0 or alert:
        print(f'{i:<6} {auc:>8.4f} {cusum.S_high:>10.3f} {cusum.S_low:>10.3f} {ewma_val:>8.4f} {"*** ALERT" if alert else ""}')

## 4. Monitoring Dashboard with Prometheus Metrics

In [ ]:
# Production monitoring setup — FastAPI + Prometheus metrics
# This shows the pattern; run in a real service for live metrics

MONITORING_FASTAPI = '''
# pip install fastapi prometheus-client evidently
from fastapi import FastAPI
from prometheus_client import Counter, Histogram, Gauge, generate_latest
from fastapi.responses import PlainTextResponse
import numpy as np
import time

app = FastAPI()

# ── Prometheus metrics ────────────────────────────────────────
PREDICTIONS = Counter("ml_predictions_total", "Total predictions", ["model", "version"])
LATENCY = Histogram("ml_prediction_latency_seconds", "Prediction latency",
                     buckets=[0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0])
SCORE_HISTOGRAM = Histogram("ml_prediction_score", "Distribution of prediction scores",
                             buckets=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
DATA_DRIFT_GAUGE = Gauge("ml_data_drift_psi", "PSI drift score per feature", ["feature"])
MODEL_AUC = Gauge("ml_model_auc", "Current model AUC-ROC", ["model", "version"])
FEATURE_MEAN = Gauge("ml_feature_mean", "Rolling mean of input feature", ["feature"])

@app.get("/metrics")
async def metrics():
    return PlainTextResponse(generate_latest())

@app.post("/predict")
async def predict(features: dict):
    t0 = time.perf_counter()

    # ... model inference ...
    score = model.predict_proba([list(features.values())])[0][1]

    latency = time.perf_counter() - t0
    PREDICTIONS.labels(model="fraud_v2", version="2.1").inc()
    LATENCY.observe(latency)
    SCORE_HISTOGRAM.observe(score)

    # Update feature means (rolling window in production)
    for feat_name, val in features.items():
        FEATURE_MEAN.labels(feature=feat_name).set(val)

    return {"score": score, "latency_ms": latency * 1000}

@app.post("/monitor/drift")
async def update_drift_scores(drift_scores: dict):
    """Called by a background drift monitoring job."""
    for feature, psi in drift_scores.items():
        DATA_DRIFT_GAUGE.labels(feature=feature).set(psi)
    return {"status": "updated"}
'''

GRAFANA_DASHBOARD = '''
Grafana dashboard panels for ML monitoring:

Row 1: Model Performance
  - AUC-ROC over time (line chart) → alert if drops > 3% from baseline
  - Precision / Recall over time
  - F1 score by cohort (mobile vs desktop, new vs returning users)

Row 2: Prediction Distribution
  - Score histogram (compare training vs current week)
  - P(score > 0.5) over time → sudden jump = feature or model issue
  - Prediction vs ground truth correlation

Row 3: Feature Health
  - PSI per feature (heatmap → red=high drift)
  - Feature mean/std over time (rolling 7-day)
  - Missing value rate per feature

Row 4: Infrastructure
  - Prediction latency P50/P95/P99
  - Request rate and error rate
  - Cache hit rate (if using prediction caching)
'''

print('Production ML Monitoring Architecture:')
print('='*55)
print()
print('Data Flow:')
print('  Predictions → FastAPI → Prometheus metrics → Grafana dashboards')
print('  Labels (delayed) → Quality evaluator → Prometheus → Grafana')
print('  Features → Drift detector → PSI/KS scores → Prometheus → Alerts')
print()
print('Alert Rules (Prometheus AlertManager):')
alerts = [
    ('PSI > 0.25 on any feature', 'Warning: data drift detected', 'Investigate data pipeline'),
    ('AUC drops > 5% in 7 days', 'Critical: model degradation', 'Trigger retraining'),
    ('P99 latency > 500ms', 'Critical: serving degraded', 'Check model server + infra'),
    ('Error rate > 1%', 'Critical: prediction failures', 'Check model + feature store'),
    ('Missing values > 10%', 'Warning: upstream data issue', 'Check ETL pipeline'),
]
for condition, label, action in alerts:
    print(f'  [{label:<35}] if {condition}')
    print(f'    → Action: {action}')
    print()
print(GRAFANA_DASHBOARD)

## 5. Automated Retraining Trigger System

In [ ]:
import numpy as np
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum

class RetrainingTrigger(Enum):
    NONE = 'none'
    SCHEDULE = 'schedule'             # Time-based: every N days
    PERFORMANCE = 'performance'        # AUC dropped below threshold
    DRIFT = 'drift'                   # Significant feature drift
    DATA_VOLUME = 'data_volume'       # Enough new labeled data accumulated

@dataclass
class RetrainingConfig:
    min_auc: float = 0.80             # Retrain if AUC drops below this
    max_psi: float = 0.25             # Retrain if PSI exceeds this
    retrain_after_days: int = 30      # Force retrain every N days
    min_new_labels: int = 500         # Min new labeled samples for data-triggered retrain
    max_daily_retrains: int = 1       # Rate limit to avoid thrashing

@dataclass
class ModelHealthState:
    model_version: str
    trained_at: datetime
    current_auc: float = 0.90
    psi_scores: dict = field(default_factory=dict)
    new_labels_since_training: int = 0
    retrains_today: int = 0

class RetrainingOrchestrator:
    """Decides when to retrain and orchestrates the pipeline."""

    def __init__(self, config: RetrainingConfig):
        self.config = config
        self.trigger_log: list[dict] = []

    def evaluate(self, state: ModelHealthState, current_time: datetime) -> tuple[RetrainingTrigger, str]:
        """Check all trigger conditions, return highest priority trigger."""
        c = self.config

        # Rate limit
        if state.retrains_today >= c.max_daily_retrains:
            return RetrainingTrigger.NONE, 'Rate limit reached — max retrains/day'

        # P1: Performance degradation (immediate action)
        if state.current_auc < c.min_auc:
            return RetrainingTrigger.PERFORMANCE, (
                f'AUC={state.current_auc:.3f} < threshold={c.min_auc}'
            )

        # P2: Significant data drift
        drifted = {f: psi for f, psi in state.psi_scores.items() if psi > c.max_psi}
        if drifted:
            return RetrainingTrigger.DRIFT, (
                f'{len(drifted)} features drifted: {drifted}'
            )

        # P3: New labeled data available
        if state.new_labels_since_training >= c.min_new_labels:
            return RetrainingTrigger.DATA_VOLUME, (
                f'{state.new_labels_since_training} new labels accumulated'
            )

        # P4: Scheduled retraining
        days_since_train = (current_time - state.trained_at).days
        if days_since_train >= c.retrain_after_days:
            return RetrainingTrigger.SCHEDULE, (
                f'{days_since_train} days since last training (limit: {c.retrain_after_days})'
            )

        return RetrainingTrigger.NONE, 'All health checks passed'

    def trigger_retrain(self, trigger: RetrainingTrigger, reason: str, state: ModelHealthState):
        """Log and initiate retraining."""
        self.trigger_log.append({'trigger': trigger, 'reason': reason, 'time': datetime.now()})
        print(f'  RETRAIN TRIGGERED [{trigger.value}]: {reason}')
        print(f'  → Submitting training job to Kubeflow/Airflow...')
        state.retrains_today += 1

# Simulate a week of monitoring
config = RetrainingConfig(min_auc=0.82, max_psi=0.25, retrain_after_days=14, min_new_labels=300)
orchestrator = RetrainingOrchestrator(config)

scenarios = [
    {'day': 1,  'auc': 0.88, 'psi': {'age': 0.05, 'income': 0.02}, 'new_labels': 50},
    {'day': 5,  'auc': 0.85, 'psi': {'age': 0.12, 'income': 0.08}, 'new_labels': 250},
    {'day': 8,  'auc': 0.81, 'psi': {'age': 0.18, 'income': 0.15}, 'new_labels': 400},  # Low AUC
    {'day': 15, 'auc': 0.86, 'psi': {'age': 0.27, 'income': 0.05}, 'new_labels': 100},  # PSI drift
]

base_time = datetime.now() - timedelta(days=20)
trained_at = base_time - timedelta(days=5)

print('Automated Retraining Decision Log:')
print('-' * 55)
for s in scenarios:
    state = ModelHealthState(
        model_version='v2.1',
        trained_at=trained_at,
        current_auc=s['auc'],
        psi_scores=s['psi'],
        new_labels_since_training=s['new_labels'],
    )
    current_time = base_time + timedelta(days=s['day'])
    trigger, reason = orchestrator.evaluate(state, current_time)
    print(f'Day {s["day"]:>2}: AUC={s["auc"]:.2f}, PSI_max={max(s["psi"].values()):.2f}, new_labels={s["new_labels"]}')
    if trigger != RetrainingTrigger.NONE:
        orchestrator.trigger_retrain(trigger, reason, state)
    else:
        print(f'  No retrain needed: {reason}')

## Exercises

1. **Windowed drift detection:** Implement a sliding window drift detector that computes PSI over a 7-day window vs a 30-day baseline. Simulate 90 days of data where drift gradually increases from day 45. Compare detection delay between PSI and KS test. Which detects the shift earlier?

2. **Cohort-level monitoring:** Extend the monitoring system to track drift separately per user cohort (mobile vs desktop, new vs returning). Show that aggregate PSI can be low while cohort-level PSI is high (Simpson's paradox). Build a function that flags this hidden drift.

3. **Evidently AI report pipeline:** Using `evidently`, create a monitoring pipeline that: (a) loads reference data from training, (b) loads 7-day production batches, (c) generates HTML reports for data quality + drift + model performance, (d) saves reports with timestamps. Schedule it with Prefect to run daily.